# Mountain-range runoff-onset anomalies

Annual anomaly tables/heatmaps per range, the regional annual anomaly maps (combined grids and the
per-region x year panels), and polar anomaly grids for selected ranges.

The combined annual grids carry the annotated 'earlier / later than median' anomaly colorbar drawn in place via
`gsro_analysis.colorbars` (since 2026-09).

In [ ]:
import os
import textwrap

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from shapely.geometry import box

from gsro_analysis import aggregate, colorbars, paths, plotting, settings, stats
from gsro_analysis.plotting import (
    count_valid_obs, create_gmba_exists_gdf, create_mini_polar_plot,
    get_sorted_ranges, label_angle, major_tick_radii,
    plot_anomaly_heatmap_panels, plot_mountain_range_anomalies,
    plot_mountain_ranges_on_map, rgrid_labels, rgrid_labels_blank,
    rgrid_vals, wrap_labels)
from gsro_analysis.stats import build_anomalies_df

In [ ]:
config = settings.load_config()  # the dataset version lives in settings.CONFIG_FILE
gmba_gdf = aggregate.load_gmba()

# the mountain-range cube: range x elevation x aspect x chili_class x water_year, bin means as
# plain variables (<var>, <var>_std, <var>_n) plus the per-range ERA5 anomaly zonal means.
mountains_full = aggregate.open_aggregate('mountain_ranges', config.version)
# the analyses' view: CHILI classes collapsed, the analyses' thresholds (>100 px per bin, a bin-year
# needs >30 % of the bin's median pixels, tropical-Andes rule), runoff_onset_elev_relative and
# runoff_onset_mean_anomaly added; aspect converted to radians for the polar panels
mountains_ds = plotting.to_polar(stats.prepare_mountain_ranges(mountains_full))
mountains_ds

## Per-range anomaly table + heatmap panels

In [ ]:
anomalies_df = build_anomalies_df(mountains_ds)
anomalies_df

In [ ]:
sns.scatterplot(data=anomalies_df,y='latitude', x='mad', hue='continent')

In [ ]:
fig = plot_anomaly_heatmap_panels(
    anomalies_df, mountains_ds['runoff_onset_mean_anomaly'].water_year.values)
fig.savefig(paths.figdir('mountain_ranges', config.version) / 'snowmelt_onset_anomalies_by_mountain_range.png')

## Regional annual anomaly maps

In [ ]:
# per-range mean anomaly per water year; a range-year is kept only if >= 10 % of the range's
# median-pixel count has data that year (the analyses' rule)
anom = stats.range_mean_anomaly(mountains_ds, min_year_fraction=0.1)
anom_df = (anom.assign_coords(GMBA_V2_ID=mountains_ds['GMBA_V2_ID'])
               .swap_dims({'mountain_range': 'GMBA_V2_ID'}).to_pandas()
               .rename(columns=lambda y: f'runoff_onset_anomaly_WY{y}'))
gmba_gdf = gmba_gdf.drop(columns=[c for c in gmba_gdf.columns if c.startswith('runoff_onset_anomaly_WY')])
gmba_gdf = gmba_gdf.merge(anom_df, left_on='GMBA_V2_ID', right_index=True, how='left')
gmba_gdf_table = gmba_gdf[['MapName'] + list(anom_df.columns)].dropna(how='all', subset=list(anom_df.columns))
gmba_gdf_table

In [ ]:
gmba_robinson_gdf = gmba_gdf.to_crs(ccrs.Robinson())
# now drop any mountain ranges where all anomaly columns are nan
gmba_robinson_gdf = gmba_robinson_gdf.dropna(subset=[col for col in gmba_gdf.columns if col.startswith('runoff_onset_anomaly_WY')], how='all')
gmba_robinson_gdf

In [ ]:
f,ax=plt.subplots(nrows=5,ncols=2,figsize=(15,20),subplot_kw={'projection': ccrs.Robinson()},layout='constrained', sharex=True, sharey=True, dpi=300) # ccrs.Robinson()

for water_year, ax in zip(mountains_ds.water_year.values, ax.flat):
    gmba_robinson_gdf.plot(ax=ax,
                                          column=f'runoff_onset_anomaly_WY{water_year}',
                                          cmap='RdBu',
                                          edgecolor='black',
                                          # now make edge size smaller
                                          linewidth=0.1,
                                        #   missing_kwds={
                                        #       "color":"lightgrey",
                                        #       "edgecolor":"black",
                                        #       "hatch":"///",
                                        #   },
                                          vmin=-30,
                                          vmax=30,
                                          transform=ccrs.Robinson(),)
    ax.set_title(f'{water_year}')

    # gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, xlocs=[-180, -120, -60, 0, 60, 120, 180], ylocs=[-60, -40, -20, 0, 20, 40, 60, 80], linestyle='--', linewidth=0.5)
    # gl.top_labels=False
    # gl.bottom_labels=True
    # gl.right_labels=False
    # gl.left_labels=True

    # ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

    #ax.add_feature(cfeature.LAND, facecolor='lightgrey')
    ax.add_feature(cfeature.LAND, facecolor='lightgrey')

    ax.add_feature(cfeature.OCEAN, facecolor='dimgray') # 'lightcyan' powderblue
    #ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')

In [ ]:
panel_dir = paths.figdir('mountain_ranges', config.version, 'regional_panels')

regions = {
    'Western North America': {
        'extent': [-170, -100, 25, 75],
        'lims': [-2207651, 1353139, -1812394, 3064937],
        'projection': ccrs.AlbersEqualArea(central_longitude=-120, central_latitude=45, 
                                         standard_parallels=(35, 65)),
        'title': 'Western North America'
    },
    'Europe': {
        'extent': [-30, 60, 30, 71],
        'lims': [-2834056, 2413016, -1914119, 2413016], # xlim right 3013016
        'projection': ccrs.AlbersEqualArea(central_longitude=20, central_latitude=50, 
                                         standard_parallels=(40, 65)),
        'title': 'Europe'
    },
    'High Mountain Asia': {
        'extent': [60, 120, 22, 50],
        'lims': [-2519204, 1819204, -1009151, 1501249],
        'projection': ccrs.AlbersEqualArea(central_longitude=90, central_latitude=32.5, 
                                         standard_parallels=(30, 45)),
        'title': 'High Mountain Asia'
    },
    'Northern Asia': {
        'extent': [90, 170, 35, 71],
        'lims': [-754768, 2221709, np.nan, np.nan], # -1654768
        'projection': ccrs.AlbersEqualArea(central_longitude=130, central_latitude=60, 
                                         standard_parallels=(55, 65)),
        'title': 'Northern Asia'
    },
    'South America': {
        'extent': [-80, -60, -55, 15],
        'lims': [-1266087, 665087, -4241836, 3774783],
        'projection': ccrs.AlbersEqualArea(central_longitude=-70, central_latitude=-20, 
                                         standard_parallels=(-45, 5)),
        'title': 'South America'
    }
}

# Recreate individual panels with consistent sizing
for region_name, region_config in regions.items():
    # Pre-process geodataframe for this region
    bbox = box(region_config['extent'][0], region_config['extent'][2], 
               region_config['extent'][1], region_config['extent'][3])
    bbox_gdf = gpd.GeoDataFrame([1], geometry=[bbox], crs='EPSG:4326')
    
    regional_gdf = gpd.sjoin(gmba_gdf, bbox_gdf, how='inner', predicate='intersects')
    regional_gdf = regional_gdf.drop(columns=['index_right'])
    regional_gdf = regional_gdf.to_crs(region_config['projection'])
    regional_gdf = regional_gdf.dropna(subset=[col for col in gmba_gdf.columns if col.startswith('runoff_onset_anomaly_WY')], how='all')
    
    # Create individual panels with exact dimensions
    for water_year in mountains_ds.water_year.values:
        # Use consistent figure size and no margins
        fig, ax = plt.subplots(figsize=(4, 4), 
                              subplot_kw={'projection': region_config['projection']})
        
        # Plot data
        regional_gdf.plot(ax=ax,
                         column=f'runoff_onset_anomaly_WY{water_year}',
                         cmap='RdBu',
                         edgecolor='white',
                         linewidth=0.2,
                         vmin=-30,
                         vmax=30,
                         missing_kwds={"color":"darkgray", "edgecolor":"white"},
                         alpha=1.0)
        
        # Add geographic features
        ax.add_feature(cfeature.LAND, facecolor='darkgrey', alpha=1)
        ax.add_feature(cfeature.OCEAN, facecolor='dimgray', alpha=1)
        ax.set_extent(region_config['extent'], crs=ccrs.PlateCarree())
        
        # Apply limits if specified
        if not np.isnan(region_config['lims'][0]):
            ax.set_xlim(left=region_config['lims'][0])
        if not np.isnan(region_config['lims'][1]):
            ax.set_xlim(right=region_config['lims'][1])
        if not np.isnan(region_config['lims'][2]):
            ax.set_ylim(bottom=region_config['lims'][2])
        if not np.isnan(region_config['lims'][3]):
            ax.set_ylim(top=region_config['lims'][3])
        
        # Remove all ticks, labels, and spines
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
            
        if water_year == config.water_years[0]:  # first panel only
            print(f'for region: {region_name}')
            left, right = ax.get_xlim()
            bottom, top = ax.get_ylim()
            print(f'x lims: {[(int(left), int(right))]}')
            print(f'ylims: {[(int(bottom), int(top))]}')

        # Set the subplot to fill the entire figure
        plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
        
        # Save with no padding
        safe_region_name = region_name.lower().replace(" ", "_")
        plt.savefig(panel_dir / f'{safe_region_name}_WY{water_year}.png', 
                   dpi=300, bbox_inches='tight', pad_inches=0)
        plt.close()
        
    print(f"Completed {region_name}")

In [ ]:
import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec

combined_dir = paths.figdir('mountain_ranges', config.version, 'regional_panels', 'combined_annual')

# Define the region order
regions = ['western_north_america', 'south_america', 'europe', 'high_mountain_asia', 'northern_asia']
region_titles = ['Western North America', 'South America', 'Europe', 'High Mountain Asia', 'Northern Asia']

# Load one set of images to calculate aspect ratios
sample_year = mountains_ds.water_year.values[0]
aspect_ratios = []

for region in regions:
    img_path = panel_dir / f'{region}_WY{sample_year}.png'
    if os.path.exists(img_path):
        img = mpimg.imread(img_path)
        aspect_ratio = img.shape[1] / img.shape[0]  # width/height
        aspect_ratios.append(aspect_ratio)
    else:
        aspect_ratios.append(1.0)  # fallback

print(f"Aspect ratios: {aspect_ratios}")

# Create the main figure
water_years = mountains_ds.water_year.values
n_years = len(water_years)
n_regions = len(regions)

# Use aspect ratios as width ratios
fig = plt.figure(figsize=(8, 17.5))
gs = GridSpec(n_years, n_regions, 
              figure=fig,
              width_ratios=aspect_ratios,
              height_ratios=[1]*n_years,
              hspace=0.0, 
              wspace=0.0)

# Load and plot all images
for year_idx, water_year in enumerate(water_years):
    for region_idx, region in enumerate(regions):
        # Load image
        img_path = panel_dir / f'{region}_WY{water_year}.png'
        
        if os.path.exists(img_path):
            img = mpimg.imread(img_path)
            
            # Create subplot
            ax = fig.add_subplot(gs[year_idx, region_idx])
            
            # Display image with equal aspect ratio
            ax.imshow(img, aspect='equal')
            ax.set_xticks([])
            ax.set_yticks([])
            
            # Add frame
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_linewidth(1)
                spine.set_color('black')
            
            # Add titles only for top row
            # if year_idx == 0:
            #     ax.set_title(region_titles[region_idx], fontsize=12, fontweight='bold')
            
            # Add water year labels only for first column
            if region_idx == 0:
                ax.set_ylabel(f'WY{water_year}')
            #     ax.text(-0.1, 0.5, f'WY{water_year}', 
            #            transform=ax.transAxes, rotation=90, 
            #            va='center', ha='center', fontsize=12, fontweight='bold')
        else:
            print(f"Warning: {img_path} not found")

# annotated anomaly colorbar under the grid, drawn natively (it was added in slides before 2026-09)
cax = fig.add_axes([0.2, 0.065, 0.6, 0.008])
colorbars.anomaly(cax, n_years, **colorbars.EMBEDDED)

# Save the combined figure
plt.savefig(combined_dir / 'global_anomalies_all_years_grid.png', 
           dpi=300, bbox_inches='tight', pad_inches=0.01)
#plt.close()

In [ ]:
import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec

combined_dir = paths.figdir('mountain_ranges', config.version, 'regional_panels', 'combined_annual')

# Define the region order
regions = ['western_north_america', 'south_america', 'europe', 'high_mountain_asia', 'northern_asia']
region_titles = ['Western North America', 'South America', 'Europe', 'High Mountain Asia', 'Northern Asia']

# Load one set of images to calculate aspect ratios and dimensions
sample_year = mountains_ds.water_year.values[0]
aspect_ratios = []
img_widths = []
img_heights = []

for region in regions:
    img_path = panel_dir / f'{region}_WY{sample_year}.png'
    if os.path.exists(img_path):
        img = mpimg.imread(img_path)
        aspect_ratio = img.shape[1] / img.shape[0]  # width/height
        aspect_ratios.append(aspect_ratio)
        img_widths.append(img.shape[1])
        img_heights.append(img.shape[0])
    else:
        aspect_ratios.append(1.0)
        img_widths.append(1200)
        img_heights.append(1200)

print(f"Aspect ratios: {aspect_ratios}")
print(f"Image heights: {img_heights}")

# Create the main figure
water_years = mountains_ds.water_year.values
n_years = len(water_years)
n_regions = len(regions)

# Calculate figure dimensions based on desired subplot sizes
# Target width per year column in inches (will be consistent across regions)
target_width_per_year = 2.0  

# Calculate height ratios based on actual image aspect ratios
# Each region should maintain its aspect ratio
height_ratios = [h / w * target_width_per_year for h, w in zip(img_heights, img_widths)]

# Make South America (index 1) have the same height as Western North America (index 0)
height_ratios[1] = height_ratios[0]

print(f"Height ratios: {height_ratios}")

# Total figure dimensions
fig_width = target_width_per_year * n_years
fig_height = sum(height_ratios)

# Create figure
fig = plt.figure(figsize=(fig_width, fig_height))
gs = GridSpec(n_regions, n_years, 
              figure=fig,
              width_ratios=[1]*n_years,  # Equal width for each year
              height_ratios=height_ratios,  # Proportional to image heights
              hspace=0.01,  # Minimal vertical spacing
              wspace=0.01)  # Minimal horizontal spacing

# Load and plot all images
for region_idx, region in enumerate(regions):
    for year_idx, water_year in enumerate(water_years):
        # Load image
        img_path = panel_dir / f'{region}_WY{water_year}.png'
        
        if os.path.exists(img_path):
            img = mpimg.imread(img_path)
            
            # Create subplot
            ax = fig.add_subplot(gs[region_idx, year_idx])
            
            # Display image with aspect='equal' to preserve aspect ratio
            # South America will naturally have horizontal whitespace
            ax.imshow(img, aspect='equal')
            ax.set_xticks([])
            ax.set_yticks([])
            
            # Add frame
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_linewidth(0.5)
                spine.set_color('black')
            
            # Add titles only for top row
            if region_idx == 0:
                ax.set_title(f'WY{water_year}', fontsize=15, pad=2)
            
            # Add region labels only for first column
            # if year_idx == 0:
            #     ax.set_ylabel(region_titles[region_idx], fontsize=8, labelpad=2, rotation=90, va='center')
        else:
            print(f"Warning: {img_path} not found")

# annotated anomaly colorbar under the grid, drawn natively
cax = fig.add_axes([0.3, 0.04, 0.4, 0.012])
colorbars.anomaly(cax, n_years, **colorbars.EMBEDDED)

# Save the combined figure
plt.savefig(combined_dir / 'global_anomalies_all_years_grid_horizontal.png', 
           dpi=300, bbox_inches='tight', pad_inches=0.02)
# plt.close()

In [ ]:
import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec
import contextily as ctx

# Load one set of images to calculate aspect ratios and dimensions (reuse from earlier)
sample_year = mountains_ds.water_year.values[0]
aspect_ratios = []
img_widths = []
img_heights = []

for region in regions:
    img_path = panel_dir / f'{region}_WY{sample_year}.png'
    if os.path.exists(img_path):
        img = mpimg.imread(img_path)
        aspect_ratio = img.shape[1] / img.shape[0]
        aspect_ratios.append(aspect_ratio)
        img_widths.append(img.shape[1])
        img_heights.append(img.shape[0])
    else:
        aspect_ratios.append(1.0)
        img_widths.append(1200)
        img_heights.append(1200)

# Calculate same dimensions as before
target_width_per_year = 2.0
height_ratios = [h / w * target_width_per_year for h, w in zip(img_heights, img_widths)]
height_ratios[1] = height_ratios[0]  # South America matches Western North America

# Region configs dictionary (from earlier in your code)
region_configs = {
    'western_north_america': {
        'extent': [-170, -100, 25, 75],
        'lims': [-2207651, 1353139, -1812394, 3064937],
        'projection': ccrs.AlbersEqualArea(central_longitude=-120, central_latitude=45, 
                                         standard_parallels=(35, 65)),
    },
    'south_america': {
        'extent': [-80, -60, -55, 15],
        'lims': [-1266087, 665087, -4241836, 3774783],
        'projection': ccrs.AlbersEqualArea(central_longitude=-70, central_latitude=-20, 
                                         standard_parallels=(-45, 5)),
    },
    'europe': {
        'extent': [-30, 60, 30, 71],
        'lims': [-2834056, 2413016, -1914119, 2413016],
        'projection': ccrs.AlbersEqualArea(central_longitude=20, central_latitude=50, 
                                         standard_parallels=(40, 65)),
    },
    'high_mountain_asia': {
        'extent': [60, 120, 22, 50],
        'lims': [-2519204, 1819204, -1009151, 1501249],
        'projection': ccrs.AlbersEqualArea(central_longitude=90, central_latitude=32.5, 
                                         standard_parallels=(30, 45)),
    },
    'northern_asia': {
        'extent': [90, 170, 35, 71],
        'lims': [-754768, 2221709, np.nan, np.nan],
        'projection': ccrs.AlbersEqualArea(central_longitude=130, central_latitude=60, 
                                         standard_parallels=(55, 65)),
    }
}

# Create context map figure with matching dimensions
fig_width = 3  # Single column, narrower than main grid
fig_height = sum(height_ratios)

fig = plt.figure(figsize=(fig_width, fig_height))
gs = GridSpec(n_regions, 1,
              figure=fig,
              height_ratios=height_ratios,
              hspace=0.01)

for idx, (region, title) in enumerate(zip(regions, region_titles)):
    region_cfg = region_configs[region]  # NOT `config` — that is the global Config
    
    # Create subplot
    ax = fig.add_subplot(gs[idx, 0], projection=region_cfg['projection'])
    
    # Set extent
    ax.set_extent(region_cfg['extent'], crs=ccrs.PlateCarree())
    
    # Prepare regional geodataframe
    bbox = box(region_cfg['extent'][0], region_cfg['extent'][2],
               region_cfg['extent'][1], region_cfg['extent'][3])
    bbox_gdf = gpd.GeoDataFrame([1], geometry=[bbox], crs='EPSG:4326')
    
    regional_gdf = gpd.sjoin(gmba_gdf, bbox_gdf, how='inner', predicate='intersects')
    regional_gdf = regional_gdf.drop(columns=['index_right'])
    regional_gdf = regional_gdf.to_crs(region_cfg['projection'])
    regional_gdf = regional_gdf.dropna(
        subset=[col for col in gmba_gdf.columns if col.startswith('runoff_onset_anomaly_WY')],
        how='all'
    )
    
    # Add Esri World Imagery basemap
    ctx.add_basemap(ax, crs=region_cfg['projection'].proj4_init,
                    source=ctx.providers.Esri.WorldImagery,
                    attribution=False)
    
    # Plot mountain ranges with thinner black outlines
    regional_gdf.plot(ax=ax, facecolor='none', edgecolor='black',
                     linewidth=0.2)
    
    # Apply x-limits from config to match main panels
    if not np.isnan(region_cfg['lims'][0]):
        ax.set_xlim(left=region_cfg['lims'][0])
    if not np.isnan(region_cfg['lims'][1]):
        ax.set_xlim(right=region_cfg['lims'][1])
    if not np.isnan(region_cfg['lims'][2]):
        ax.set_ylim(bottom=region_cfg['lims'][2])
    if not np.isnan(region_cfg['lims'][3]):
        ax.set_ylim(top=region_cfg['lims'][3])
    
    # Remove tick labels
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Add frame to match main grid with thinner lines
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.3)
        spine.set_color('black')

plt.savefig(combined_dir / 'regional_context_maps.png',
           dpi=300, bbox_inches='tight', pad_inches=0.02)
#plt.close()

## Polar anomaly grids for selected ranges

In [ ]:
mountain_params = {
    'Olympic Mountains': {
        'runoff_cmap_min': 80,
        'runoff_cmap_max': 220,
        'bottom_r': 2000,
        'top_r': 0
    },
    'Sierra Nevada': {
        'runoff_cmap_min': 150,
        'runoff_cmap_max': 250,
        'bottom_r': 4000,
        'top_r': 0
    },
    'Cascade Range': {
        'runoff_cmap_min': 120,
        'runoff_cmap_max': 220,
        'bottom_r': 3000,
        'top_r': 0
    },
    'Brooks Range': {
        'runoff_cmap_min': 200,
        'runoff_cmap_max': 275,
        'bottom_r': 3000,
        'top_r': 0
    },
    'South-Central Alaska': {
        'runoff_cmap_min': 180,
        'runoff_cmap_max': 300,
        'bottom_r': 3000,
        'top_r': 0
    },
    'Oregon Coast Range': {
        'runoff_cmap_min': 100,
        'runoff_cmap_max': 160,
        'bottom_r': 2000,
        'top_r': 0
    },
    'Great Basin Ranges': {
        'runoff_cmap_min': 100,
        'runoff_cmap_max': 330,
        'bottom_r': 4000,
        'top_r': 1000
    }
}

In [ ]:
# display ranges present in this version of the cube (a missing range is skipped, not an error)
mountain_range_subset = [r for r in ['Brooks Range', 'South-Central Alaska', 'Olympic Mountains',
                                     'Cascade Range', 'Sierra Nevada', 'Great Basin Ranges']
                         if r in mountains_ds.mountain_range.values] or list(mountains_ds.mountain_range.values[:6])   # fallback: first ranges present
location = 'Sierra Nevada' if 'Sierra Nevada' in mountains_ds.mountain_range.values else str(mountains_ds.mountain_range.values[0])
params = mountain_params.get(location, {'runoff_cmap_min': 200, 'runoff_cmap_max': 275, 'bottom_r': 8000, 'top_r': 0})
runoff_cmap_min, runoff_cmap_max = params['runoff_cmap_min'], params['runoff_cmap_max']
bottom_r, top_r = params['bottom_r'], params['top_r']
n_years = len(mountains_ds.water_year)
mountain_range_subset

In [ ]:
#f,ax=plt.subplots(figsize=(8,7),subplot_kw=dict(projection='polar'))
grid = mountains_ds.sel(mountain_range=mountain_range_subset)['runoff_onset_anomaly'].plot(row='mountain_range',col='water_year',vmin=-30,vmax=30, yincrease=False, cmap='RdBu', subplot_kws=dict(projection='polar'),add_colorbar=False,cbar_kwargs={"label": "[Days]"})
grid.fig.dpi = 300

top_r = 0
bottom_r = 4000
for i,ax in enumerate(grid.axs.flat):

    location = mountain_range_subset[i // n_years]

    # bottom_r = mountain_params[location]['bottom_r']
    # top_r = mountain_params[location]['top_r']

    ax.set_facecolor('darkgrey')
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)

    for major_tick_radius in major_tick_radii:
        ax.plot(np.linspace(0, 2*np.pi, 100), np.ones(100)*major_tick_radius, color='black', linestyle='-', alpha=1, linewidth=1)
        
    ax.scatter(x=0,y=bottom_r,marker='.',color='black',s=100)

    ax.set_xlabel('')
    ax.set_ylabel('')
    #if i % 10 != 0:
        #ax.set_title(location)
    ax.set_title('')
    #ax.set_thetagrids([0,45,90,135,180,225,270,315,360],labels=['N','NE','E','SE','S','SW','W','NW','N'])
    ax.set_thetagrids([0,45,90,135,180,225,270,315,360],labels=['','','','','','','','',''])

    ax.set_rgrids(rgrid_vals,labels=rgrid_labels_blank,angle=label_angle,fontsize=7,ha='left',va='bottom',zorder=0)

    ax.xaxis.grid(True, which="both", linestyle=":", color='gray',alpha=0.8, linewidth=1)
    ax.yaxis.grid(True, which="both", linestyle=":", color='black',alpha=1, linewidth=1)
    [x.set_linewidth(1.2) for x in ax.spines.values()]
    ax.set_thetamin(0)
    ax.set_thetamax(360)
    ax.set_rlim(bottom=bottom_r, top=top_r)
    #ax.set_rlim(bottom=bottom_r, top=top_r)
        #ax.set_rlim(bottom=bottom_r, top=top_r)

# for i, location in enumerate(mountain_range_subset):
#     bottom_r = mountain_params[location]['bottom_r']
#     top_r = mountain_params[location]['top_r']
#     grid.axs.flat[i].set_rlim(bottom=bottom_r, top=top_r)
    


#grid.fig.subplots_adjust(wspace=0.05,hspace=-0.8,left=0,right=1,top=0.99,bottom=0.01)
#grid.fig.subplots_adjust(wspace=0)
grid.fig.tight_layout()

In [ ]:
n_years = len(mountains_ds.water_year)
n_cols = len(mountains_ds.water_year)  # one column per water year
n_rows = 2   # Runoff Onset on top, Anomaly on bottom

# Create figure with gridspec to accommodate colorbars and labels
fig = plt.figure(figsize=(15, 3), dpi=300)
gs = fig.add_gridspec(n_rows, n_cols + 2, width_ratios=[0.3] + [1]*n_cols + [0.05])

# Create axes for labels on the far left
ax_label_onset = fig.add_subplot(gs[0, 0])
ax_label_anomaly = fig.add_subplot(gs[1, 0])

# Hide axes and add text labels
ax_label_onset.axis('off')
ax_label_anomaly.axis('off')
ax_label_onset.text(0.6, 0.5, 'Runoff onset', rotation=90,
                    va='center', ha='center', fontsize=12)
ax_label_anomaly.text(0.6, 0.5, 'Runoff anomaly', rotation=90,
                      va='center', ha='center', fontsize=12)

# Create axes for plots
axes_top = [fig.add_subplot(gs[0, i+1], projection='polar') for i in range(n_cols)]
axes_bottom = [fig.add_subplot(gs[1, i+1], projection='polar') for i in range(n_cols)]
cax_onset = fig.add_subplot(gs[0, -1])
cax_anomaly = fig.add_subplot(gs[1, -1])

# Extract bin edges from the interval index
# aspect_bin_edges = np.array(
#     [interval.left for interval in mountains_ds.aspect.values] +
#     [mountains_ds.aspect.values[-1].right]
# )
# dem_bin_edges = np.array(
#     [interval.left for interval in mountains_ds.elevation.values] +
#     [mountains_ds.elevation.values[-1].right]
# )


# Create meshgrid for plotting
theta, r = np.meshgrid(mountains_ds.aspect.values, mountains_ds.elevation.values)

# Plot data for each year
for idx, year in enumerate(mountains_ds.water_year.values):
    if idx >= n_cols:
        break  # Avoid exceeding the number of columns

    # Select data
    data_onset = mountains_ds.sel(mountain_range=location)['runoff_onset'].sel(water_year=year).values
    data_anomaly = mountains_ds.sel(mountain_range=location)['runoff_onset_anomaly'].sel(water_year=year).values

    # Ensure data dimensions match pcolormesh requirements
    # expected_shape = (len(mountains_ds.elevation.values) - 1, len(mountains_ds.aspect.values) - 1)
    # data_onset = data_onset[:expected_shape[0], :expected_shape[1]]
    # data_anomaly = data_anomaly[:expected_shape[0], :expected_shape[1]]

    # Plotting runoff onset on the top row
    ax_onset = axes_top[idx]
    im_onset = ax_onset.pcolormesh(theta, r, data_onset,
                                   vmin=runoff_cmap_min, vmax=runoff_cmap_max, cmap='viridis', shading='auto')

    # Plotting runoff onset anomaly on the bottom row
    ax_anomaly = axes_bottom[idx]
    im_anomaly = ax_anomaly.pcolormesh(theta, r, data_anomaly,
                                       vmin=-30, vmax=30, cmap='RdBu', shading='auto')

    # Formatting axes
    for ax, title in zip([ax_onset, ax_anomaly], [f'WY {year}', '']):
        ax.set_facecolor('darkgrey')
        ax.set_theta_zero_location('N')
        ax.set_theta_direction(-1)

        for major_tick_radius in major_tick_radii:
            ax.plot(
                np.linspace(0, 2 * np.pi, 100),
                np.ones(100) * major_tick_radius,
                color='black', linestyle='-', alpha=1, linewidth=1
            )

        ax.scatter(x=0, y=bottom_r, marker='.', color='black', s=100)
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_title(title)
        ax.set_thetagrids(
            [0, 45, 90, 135, 180, 225, 270, 315, 360],
            labels=[''] * 9
        )
        ax.set_rgrids(
            rgrid_vals, labels=rgrid_labels_blank,
            angle=label_angle, fontsize=7,
            ha='left', va='bottom', zorder=0
        )
        ax.xaxis.grid(True, which="both", linestyle=":", color='gray',
                      alpha=0.8, linewidth=1)
        ax.yaxis.grid(True, which="both", linestyle=":", color='black',
                      alpha=1, linewidth=1)
        for spine in ax.spines.values():
            spine.set_linewidth(1.2)
        ax.set_thetamin(0)
        ax.set_thetamax(360)
        ax.set_rlim(bottom=bottom_r, top=top_r)





fig.suptitle(f'Runoff onset and anomaly\n{location}', fontsize=16, y=1.3)

# Adjust layout
fig.subplots_adjust(
    wspace=0.05, hspace=0.01,
    left=0, right=1,
    top=1, bottom=0
)

cbar_width = 0.01  # Width of colorbars
cbar_spacing = 0.1  # Space between the plotting area and colorbars

# Calculate left positions for colorbars
cbar_left = 0.9 + cbar_spacing  # Start after the plotting area and spacing

# Set positions for colorbars
cax_onset.set_position([cbar_left, 0.55, cbar_width, 0.4])    # [left, bottom, width, height]
cax_anomaly.set_position([cbar_left, 0.05, cbar_width, 0.4])  # Adjust positions as needed

# Clear colorbar axes (optional, to ensure they are empty)
cax_onset.cla()
cax_anomaly.cla()

# Add colorbars for each row
im_cbar_onset = fig.colorbar(im_onset, cax=cax_onset,
                             label='[DOWY]', orientation='vertical')
im_cbar_anomaly = fig.colorbar(im_anomaly, cax=cax_anomaly,
                               label='[Days]', orientation='vertical')

#fig.tight_layout()

In [ ]:
# Create figure with gridspec
n_years = len(mountains_ds.water_year)
fig = plt.figure(figsize=(3, 1.4 * n_years), dpi=300)
gs = fig.add_gridspec(2 * n_years + 2, 2, height_ratios=[1] * (2 * n_years) + [0.5] * 2)

# Create axes for plots
axes_left = [fig.add_subplot(gs[i:i+2, 0], projection='polar') for i in range(0, 2 * n_years, 2)]
axes_right = [fig.add_subplot(gs[i:i+2, 1], projection='polar') for i in range(0, 2 * n_years, 2)]
axes_onset = axes_left
axes_anomaly = axes_right

# Create colorbar axes
cax_onset = fig.add_subplot(gs[2 * n_years, 0])
cax_anomaly = fig.add_subplot(gs[2 * n_years, 1])

# Extract bin edges and create meshgrid (same as before)
# aspect_bin_edges = np.array(
#     [interval.left for interval in mountains_ds.aspect.values] +
#     [mountains_ds.aspect.values[-1].right]
# )
# dem_bin_edges = np.array(
#     [interval.left for interval in mountains_ds.elevation.values] +
#     [mountains_ds.elevation.values[-1].right]
#)
theta, r = np.meshgrid(mountains_ds.aspect.values, mountains_ds.elevation.values)

# Plot data for each year
for idx, year in enumerate(mountains_ds.water_year.values):
    if idx >= n_years:
        break

    data_onset = mountains_ds.sel(mountain_range=location)['runoff_onset'].sel(water_year=year).values
    data_anomaly = mountains_ds.sel(mountain_range=location)['runoff_onset_anomaly'].sel(water_year=year).values

    # expected_shape = (len(dem_bin_edges) - 1, len(aspect_bin_edges) - 1)
    # data_onset = data_onset[:expected_shape[0], :expected_shape[1]]
    # data_anomaly = data_anomaly[:expected_shape[0], :expected_shape[1]]

    ax_onset = axes_onset[idx]
    ax_anomaly = axes_anomaly[idx]
    
    im_onset = ax_onset.pcolormesh(theta, r, data_onset,
                                  vmin=runoff_cmap_min, vmax=runoff_cmap_max, cmap='viridis', shading='auto')
    im_anomaly = ax_anomaly.pcolormesh(theta, r, data_anomaly,
                                      vmin=-30, vmax=30, cmap='RdBu', shading='auto')

    # Format axes
    for ax in [ax_onset, ax_anomaly]:
        ax.set_facecolor('darkgrey')
        ax.set_theta_zero_location('N')
        ax.set_theta_direction(-1)

        for major_tick_radius in major_tick_radii:
            ax.plot(np.linspace(0, 2*np.pi, 100), np.ones(100)*major_tick_radius,
                   color='black', linestyle='-', alpha=1, linewidth=1)

        ax.scatter(x=0, y=bottom_r, marker='.', color='black', s=100)
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_thetagrids([0,45,90,135,180,225,270,315,360], 
                         labels=['','','','','','','','',''])
        ax.set_rgrids(rgrid_vals, labels=rgrid_labels_blank,
                     angle=label_angle, fontsize=7, ha='left', va='bottom', zorder=0)
        
        ax.xaxis.grid(True, which="both", linestyle=":", color='gray', alpha=0.8, linewidth=1)
        ax.yaxis.grid(True, which="both", linestyle=":", color='black', alpha=1, linewidth=1)
        [x.set_linewidth(1.2) for x in ax.spines.values()]
        ax.set_thetamin(0)
        ax.set_thetamax(360)
        ax.set_rlim(bottom=bottom_r, top=top_r)
    
    # Add water year label only to left column
    if ax_onset:
        ax_onset.text(-0.2, 0.5, f'WY {year}', transform=ax_onset.transAxes,
                     va='center', ha='right', fontsize=10, rotation=90)


#fig.subplots_adjust(left=0,right=1,wspace=0)

# Set positions for colorbars
cax_onset.set_position([0.137, 0.14, 0.32, 0.01])    # [left, bottom, width, height]
cax_anomaly.set_position([0.565, 0.14, 0.32, 0.01])  # Adjust positions as needed


# Add colorbars
fig.colorbar(im_onset, cax=cax_onset, orientation='horizontal', label='[DOWY]')
fig.colorbar(im_anomaly, cax=cax_anomaly, orientation='horizontal', label='[Days]', ticks=[-30,0,30])

# Add column labels
fig.text(0.29, 0.895, 'Runoff onset', ha='center', va='top', fontsize=12)
fig.text(0.75, 0.895, 'Runoff anomaly', ha='center', va='top', fontsize=12)

# Title
fig.suptitle(f'{location}', fontsize=16, y=0.92)
# Adjust spacing
#fig.tight_layout()
# fig.subplots_adjust(hspace=0.1, wspace=0.05,
#                     left=0, right=1,
#                     top=1, bottom=0)

In [ ]:
mountain_range_subset = ['Brooks Range', 
                   'Alaska Intermountain Ranges',
                   'Yukon Intermountain Ranges',
                   'Alaska Range',
                   'South-Central Alaska',
                   'Saint Elias Mountains',
                   'Aleutian Ranges',
                   'Far Northern Rockies',
                   'Central Labrador Ranges', 
                   'Olympic Mountains', 
                   'Cascade Range',
                   #'Oregon Coast Range', 
                   'Sierra Nevada',
                   'Great Basin Ranges']


mountain_range_subset = ['Brooks Range', 
                   #'Alaska Intermountain Ranges',
                   #'Yukon Intermountain Ranges',
                   #'Alaska Range',
                   'South-Central Alaska',
                   'Saint Elias Mountains',
                   #'Aleutian Ranges',
                   #'Far Northern Rockies',
                   #'Central Labrador Ranges', 
                   #'British Columbia Interior',
                   'Coast Mountains',
                   'Canadian Rockies',
                   #'Columbia Mountains',
                   #'Insular Mountains',
                   #'Olympic Mountains',
                   'Central Montana Rocky Mountains', 
                   'Cascade Range',
                   #'Idaho-Bitterroot Rocky Mountains',
                   #'Columbia Plateau',
                   'Greater Yellowstone Rockies',
                   #'Klamath Mountains',
                   #'Western Rocky Mountains',
                   #'Oregon Coast Range', 
                   'Great Basin Ranges',
                   'Southern Rocky Mountains',
                   'Sierra Nevada',
                   #'Colorado Plateau',
                   #'Southwest Basins and Ranges',
                   ]

# keep the display ranges present in this version of the cube (fallback: the first six present)
mountain_range_subset = [r for r in mountain_range_subset if r in mountains_ds.mountain_range.values] or list(mountains_ds.mountain_range.values[:6])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Example usage:
fig = plot_mountain_range_anomalies(mountains_ds, [r for r in mountain_range_subset if r in mountains_ds.mountain_range.values], mountain_params)

fig.savefig(paths.figdir('mountain_ranges', config.version) / 'median_runoff_and_anomaly_subset.png',dpi=300)